In [1]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Load the existing classification data to understand the structure
df = pd.read_csv('sk_covariance_classification.csv')
print("Existing SK Covariance Classification Data:")
print(df)
print("\nKey metrics for L_DH (F5):")
ldh_row = df[df['Function_ID'] == 'F5']
print(ldh_row[['Function_Name', 'CG_N', 'Sum_Offdiag_N1', 'Sum_Offdiag_N2']])


Existing SK Covariance Classification Data:
 Function_ID Function_Name RH_Status CS_N CG_N \
0 F1 Riemann ζ True -2.145868 -48.166024 
1 F2 L(χ₄) real mod 5 True -0.911860 -141.092943 
2 F3 L(χ) complex mod 5 True -1.392495 -73.487153 
3 F4 Random multiplicative NaN -0.698308 -28.874906 
4 F5 Davenport-Heilbronn False -1.015625 -58.848451 
5 F6 Liouville L(s,λ) NaN -1.201193 -1.543072 
6 F7 Möbius L(s,μ) NaN -0.480808 251.765687 
7 F8 Fully random NaN -0.284477 -119.689792 

 Sum_Offdiag_N1 Sum_Offdiag_N2 N_Negative_N1 N_Negative_N2 \
0 -1.382172 -2.047909 22 20 
1 -0.373290 -0.899975 16 24 
2 -0.798141 -1.384673 18 28 
3 -0.443553 -0.571629 16 16 
4 -0.615537 -0.977771 20 20 
5 -1.089483 -1.106294 20 16 
6 -0.085233 0.129354 14 10 
7 -0.025751 -0.056572 14 18 

 Sum_Negative_N1 Sum_Negative_N2 Distance_to_RH_centroid 
0 -1.408797 -2.145868 0.770854 
1 -0.375498 -0.911860 0.782948 
2 -0.800255 -1.392495 0.167725 
3 -0.478986 -0.698308 0.980324 
4 -0.622760 -1.015625 0.548983 
5 -1.1138

In [2]:

# Analysis Plan:
# 1. Implement the canonical "historical" L_DH approximation (real-valued periodic)
# 2. Create two perturbed variants with κ' = κ(1±0.05)
# 3. Generate time series of Re(S_k(t;N)) for k=1..6 at N∈{10^5, 10^6, 2×10^6}
# 4. Compute 6x6 covariance matrices and sum of 15 off-diagonal terms at each N
# 5. Calculate CG_N for intervals [10^5, 10^6] and [10^6, 2×10^6]
# 6. Check for sign reversal pattern

# First, let's implement the historical L_DH approximation
# From the literature, the historical L_DH is a real-valued periodic approximation
# The canonical form uses a Fourier-like series with specific coefficient structure

def historical_ldh_coefficients(n_max, kappa=None):
 """
 Generate coefficients for the historical L_DH approximation.
 This is a real-valued periodic approximation that has been validated.
 
 The historical implementation uses a specific periodic structure with
 alternating signs and a scaling parameter kappa.
 """
 # Default kappa value based on canonical definition
 if kappa is None:
 # The canonical value from Davenport-Heilbronn analysis
 kappa = 2 * np.pi
 
 n = np.arange(1, n_max + 1)
 
 # Historical L_DH uses a periodic coefficient structure
 # a_n = (-1)^(n+1) * kappa * sin(kappa * log(n)) for the historical approximation
 # This is real-valued and has been successfully validated
 coeffs = (-1)**(n + 1) * np.sin(kappa * np.log(n))
 
 return coeffs

# Test the coefficient generation
n_test = 20
coeffs_test = historical_ldh_coefficients(n_test)
print("First 20 coefficients of historical L_DH:")
print(coeffs_test)
print(f"\nCoefficient statistics:")
print(f"Mean: {np.mean(coeffs_test):.6f}")
print(f"Std: {np.std(coeffs_test):.6f}")
print(f"Min: {np.min(coeffs_test):.6f}")
print(f"Max: {np.max(coeffs_test):.6f}")


First 20 coefficients of historical L_DH:
[ 0. 0.93687365 0.58070898 -0.65518963 -0.6346988 0.96577475
 -0.33335186 -0.47867584 0.94552339 -0.94591204 0.5984323 -0.09469223
 -0.39685584 0.7667243 -0.96546387 0.98994496 -0.86640212 0.63562247
 -0.34205241 0.0268117 ]

Coefficient statistics:
Mean: 0.036656
Std: 0.684795
Min: -0.965464
Max: 0.989945


In [3]:

def kahan_sum(values):
 """
 Kahan compensated summation for high-precision accumulation.
 """
 total = 0.0
 c = 0.0 # Compensation for lost low-order bits
 
 for value in values:
 y = value - c
 t = total + y
 c = (t - total) - y
 total = t
 
 return total

def compute_dirichlet_partial_sum(coeffs, t, use_kahan=True):
 """
 Compute Dirichlet partial sum D_F(t; N) = Σ_{n≤N} a_n(F) / n^(1/2+it)
 
 Parameters:
 -----------
 coeffs : array
 Coefficient sequence a_n for n=1..N
 t : float
 Ordinate value
 use_kahan : bool
 Whether to use Kahan compensated summation
 
 Returns:
 --------
 complex
 The partial sum value
 """
 N = len(coeffs)
 n = np.arange(1, N + 1)
 
 # Compute n^(1/2 + it) = n^(1/2) * n^(it) = sqrt(n) * exp(it*log(n))
 # Real part: a_n / sqrt(n) * cos(t * log(n))
 # Imag part: a_n / sqrt(n) * sin(t * log(n))
 
 sqrt_n = np.sqrt(n)
 log_n = np.log(n)
 
 real_terms = coeffs / sqrt_n * np.cos(t * log_n)
 imag_terms = coeffs / sqrt_n * np.sin(t * log_n)
 
 if use_kahan:
 real_sum = kahan_sum(real_terms)
 imag_sum = kahan_sum(imag_terms)
 else:
 real_sum = np.sum(real_terms)
 imag_sum = np.sum(imag_terms)
 
 return real_sum + 1j * imag_sum

# Test the partial sum computation
coeffs = historical_ldh_coefficients(10000)
t_test = 1000.0
D_test = compute_dirichlet_partial_sum(coeffs, t_test)
print(f"Test partial sum at t={t_test}, N=10000:")
print(f"D(t;N) = {D_test.real:.6f} + {D_test.imag:.6f}i")
print(f"|D(t;N)| = {np.abs(D_test):.6f}")


Test partial sum at t=1000.0, N=10000:
D(t;N) = 1.984081 + 1.793729i
|D(t;N)| = 2.674704


In [4]:

def compute_sk_values(coeffs, t, k_max=6, use_kahan=True):
 """
 Compute S_k(t; N) = Σ_{n≤N} a_n(F) / n^(k/2 + it) for k=1..k_max
 
 Parameters:
 -----------
 coeffs : array
 Coefficient sequence a_n for n=1..N
 t : float
 Ordinate value
 k_max : int
 Maximum k value (default 6)
 use_kahan : bool
 Whether to use Kahan compensated summation
 
 Returns:
 --------
 dict
 Dictionary with keys 'S1' through 'Sk_max' containing complex values
 """
 N = len(coeffs)
 n = np.arange(1, N + 1)
 log_n = np.log(n)
 
 sk_values = {}
 
 for k in range(1, k_max + 1):
 # Compute n^(k/2 + it) = n^(k/2) * exp(it*log(n))
 n_power = n**(k/2)
 
 real_terms = coeffs / n_power * np.cos(t * log_n)
 imag_terms = coeffs / n_power * np.sin(t * log_n)
 
 if use_kahan:
 real_sum = kahan_sum(real_terms)
 imag_sum = kahan_sum(imag_terms)
 else:
 real_sum = np.sum(real_terms)
 imag_sum = np.sum(imag_terms)
 
 sk_values[f'S{k}'] = real_sum + 1j * imag_sum
 
 return sk_values

# Test S_k computation
sk_test = compute_sk_values(coeffs, t_test, k_max=6)
print(f"S_k values at t={t_test}, N=10000:")
for k in range(1, 7):
 sk = sk_test[f'S{k}']
 print(f"S{k}: {sk.real:.6f} + {sk.imag:.6f}i (|S{k}| = {np.abs(sk):.6f})")


S_k values at t=1000.0, N=10000:
S1: 1.984081 + 1.793729i (|S1| = 2.674704)
S2: 0.139405 + 0.606544i (|S2| = 0.622358)
S3: -0.017300 + 0.305043i (|S3| = 0.305533)
S4: -0.036043 + 0.195752i (|S4| = 0.199042)
S5: -0.035448 + 0.135955i (|S5| = 0.140500)
S6: -0.030219 + 0.096489i (|S6| = 0.101110)


In [5]:

def generate_sk_timeseries(kappa, N, t_range, n_points, k_max=6, use_kahan=True):
 """
 Generate time series of Re(S_k(t;N)) for k=1..k_max over a range of t values.
 
 Parameters:
 -----------
 kappa : float
 Scaling parameter for the L_DH function
 N : int
 Truncation depth
 t_range : tuple
 (t_min, t_max) range for t values
 n_points : int
 Number of t points to sample
 k_max : int
 Maximum k value (default 6)
 use_kahan : bool
 Whether to use Kahan compensated summation
 
 Returns:
 --------
 dict
 Dictionary with keys:
 - 't_values': array of t values
 - 'Re_S1' through 'Re_S6': arrays of real parts of S_k
 """
 # Generate coefficients with specified kappa
 coeffs = historical_ldh_coefficients(N, kappa=kappa)
 
 # Generate t values
 t_values = np.linspace(t_range[0], t_range[1], n_points)
 
 # Initialize storage for Re(S_k) values
 result = {'t_values': t_values}
 for k in range(1, k_max + 1):
 result[f'Re_S{k}'] = np.zeros(n_points)
 
 # Compute S_k for each t value
 for i, t in enumerate(t_values):
 sk_vals = compute_sk_values(coeffs, t, k_max=k_max, use_kahan=use_kahan)
 for k in range(1, k_max + 1):
 result[f'Re_S{k}'][i] = sk_vals[f'S{k}'].real
 
 return result

# Test with small N to verify the function works
print("Testing time series generation with N=10000, 100 points...")
test_series = generate_sk_timeseries(kappa=2*np.pi, N=10000, t_range=(10000, 11000), 
 n_points=100, k_max=6)
print(f"Generated {len(test_series['t_values'])} time points")
print(f"\nSample Re(S_k) values at first few t points:")
for i in range(3):
 print(f"t={test_series['t_values'][i]:.2f}:")
 for k in range(1, 7):
 print(f" Re(S{k}) = {test_series[f'Re_S{k}'][i]:.6f}")


Testing time series generation with N=10000, 100 points...


Generated 100 time points

Sample Re(S_k) values at first few t points:
t=10000.00:
 Re(S1) = -0.340043
 Re(S2) = 0.192891
 Re(S3) = 0.131343
 Re(S4) = 0.084686
 Re(S5) = 0.057178
 Re(S6) = 0.040090
t=10010.10:
 Re(S1) = -1.588109
 Re(S2) = -0.342189
 Re(S3) = -0.161091
 Re(S4) = -0.088648
 Re(S5) = -0.052936
 Re(S6) = -0.033756
t=10020.20:
 Re(S1) = 1.045624
 Re(S2) = -0.173359
 Re(S3) = -0.207004
 Re(S4) = -0.154775
 Re(S5) = -0.112278
 Re(S6) = -0.081490


In [6]:

def compute_covariance_matrix(timeseries_data, k_max=6):
 """
 Compute the 6x6 covariance matrix for Re(S_k) time series.
 
 Parameters:
 -----------
 timeseries_data : dict
 Dictionary containing 'Re_S1' through 'Re_S6' arrays
 k_max : int
 Maximum k value (default 6)
 
 Returns:
 --------
 ndarray
 6x6 covariance matrix
 """
 # Stack the Re(S_k) arrays into a matrix (6 x n_points)
 sk_matrix = np.array([timeseries_data[f'Re_S{k}'] for k in range(1, k_max + 1)])
 
 # Compute covariance matrix (6 x 6)
 cov_matrix = np.cov(sk_matrix)
 
 return cov_matrix

def compute_offdiag_sum(cov_matrix):
 """
 Compute the sum of all off-diagonal elements in the covariance matrix.
 
 For a 6x6 matrix, there are 15 unique off-diagonal pairs.
 """
 k_max = cov_matrix.shape[0]
 offdiag_sum = 0.0
 
 for i in range(k_max):
 for j in range(i + 1, k_max):
 # Sum both symmetric elements (they should be equal for covariance matrix)
 offdiag_sum += cov_matrix[i, j] + cov_matrix[j, i]
 
 return offdiag_sum

# Test covariance computation
cov_test = compute_covariance_matrix(test_series)
offdiag_sum_test = compute_offdiag_sum(cov_test)

print("Test 6x6 Covariance Matrix:")
print(cov_test)
print(f"\nSum of off-diagonal terms: {offdiag_sum_test:.6f}")
print(f"Number of unique off-diagonal pairs: 15")
print(f"Average off-diagonal covariance: {offdiag_sum_test / 30:.6f}") # Divide by 30 since we count each pair twice


Test 6x6 Covariance Matrix:
[[1.42309145 0.30152457 0.13042965 0.07152568 0.0432141 0.02745481]
 [0.30152457 0.1620316 0.0966888 0.06183758 0.0408818 0.02753193]
 [0.13042965 0.0966888 0.06366763 0.04275935 0.02909415 0.01996083]
 [0.07152568 0.06183758 0.04275935 0.02953635 0.02046095 0.01420916]
 [0.0432141 0.0408818 0.02909415 0.02046095 0.01434472 0.01004484]
 [0.02745481 0.02753193 0.01996083 0.01420916 0.01004484 0.00707529]]

Sum of off-diagonal terms: 1.875236
Number of unique off-diagonal pairs: 15
Average off-diagonal covariance: 0.062508


In [7]:

def compute_cg_n(sum_N1, sum_N2):
 """
 Compute the Cancellation Growth metric CG_N as percentage change.
 
 CG_N = 100 * (Sum_N2 - Sum_N1) / Sum_N1
 """
 if sum_N1 == 0:
 return np.nan
 return 100 * (sum_N2 - sum_N1) / sum_N1

# Now let's implement the full analysis pipeline
# We'll start with a modest number of points due to computational constraints

print("="*80)
print("ANALYSIS PLAN")
print("="*80)
print()
print("1. Generate perturbed L_DH functions:")
print(" - L_DH^(+): κ' = 2π × 1.05")
print(" - L_DH^(-): κ' = 2π × 0.95")
print()
print("2. For each perturbed function, generate time series:")
print(" - N values: 10^5, 10^6, 2×10^6")
print(" - t range: [10000, 20000]")
print(" - Number of t points: 1500 (based on computational constraints)")
print(" - Use Kahan compensated summation")
print()
print("3. Compute covariance matrices and CG_N metrics:")
print(" - Interval 1: [10^5, 10^6]")
print(" - Interval 2: [10^6, 2×10^6]")
print()
print("4. Check for sign reversal pattern")
print("="*80)


ANALYSIS PLAN

1. Generate perturbed L_DH functions:
 - L_DH^(+): κ' = 2π × 1.05
 - L_DH^(-): κ' = 2π × 0.95

2. For each perturbed function, generate time series:
 - N values: 10^5, 10^6, 2×10^6
 - t range: [10000, 20000]
 - Number of t points: 1500 (based on computational constraints)
 - Use Kahan compensated summation

3. Compute covariance matrices and CG_N metrics:
 - Interval 1: [10^5, 10^6]
 - Interval 2: [10^6, 2×10^6]

4. Check for sign reversal pattern


In [8]:

# Define parameters
kappa_base = 2 * np.pi
kappa_plus = kappa_base * 1.05
kappa_minus = kappa_base * 0.95

N_values = [10**5, 10**6, 2*10**6]
t_range = (10000, 20000)
n_points = 1500

print(f"κ_base = {kappa_base:.6f}")
print(f"κ_plus (+5%) = {kappa_plus:.6f}")
print(f"κ_minus (-5%) = {kappa_minus:.6f}")
print()
print(f"N values: {N_values}")
print(f"t range: {t_range}")
print(f"Number of t points: {n_points}")
print()
print("Starting computations...")
print("This will take some time due to the large N values...")


κ_base = 6.283185
κ_plus (+5%) = 6.597345
κ_minus (-5%) = 5.969026

N values: [100000, 1000000, 2000000]
t range: (10000, 20000)
Number of t points: 1500

Starting computations...
This will take some time due to the large N values...


In [9]:

# The computation is taking too long. Let's optimize by:
# 1. Using fully vectorized operations (no Kahan at all)
# 2. Reducing number of t points
# 3. Computing all S_k values simultaneously

def generate_sk_timeseries_vectorized(kappa, N, t_values, k_max=6):
 """
 Fully vectorized version of S_k time series generation.
 Much faster but slightly less precise than Kahan summation.
 
 Parameters:
 -----------
 kappa : float
 Scaling parameter for the L_DH function
 N : int
 Truncation depth
 t_values : array
 Array of t values
 k_max : int
 Maximum k value (default 6)
 
 Returns:
 --------
 dict
 Dictionary with Re_S1 through Re_S6 arrays
 """
 # Generate coefficients
 n = np.arange(1, N + 1, dtype=np.float64)
 coeffs = (-1)**(n + 1) * np.sin(kappa * np.log(n))
 
 # Precompute log(n) for all n
 log_n = np.log(n)
 
 # Initialize result storage
 result = {'t_values': t_values}
 for k in range(1, k_max + 1):
 result[f'Re_S{k}'] = np.zeros(len(t_values))
 
 # Compute for each k
 for k in range(1, k_max + 1):
 # Precompute n^(-k/2)
 n_power_inv = n**(-k/2)
 
 # For each t, compute sum
 for i, t in enumerate(t_values):
 # Re(S_k) = Σ a_n / n^(k/2) * cos(t * log(n))
 result[f'Re_S{k}'][i] = np.sum(coeffs * n_power_inv * np.cos(t * log_n))
 
 return result

# Test with reduced parameters
print("Testing optimized vectorized computation...")
print("Using N=10^5 with 100 t points as a test...")

t_test_vals = np.linspace(10000, 11000, 100)
start = time.time()
test_result = generate_sk_timeseries_vectorized(2*np.pi, 10**5, t_test_vals, k_max=6)
elapsed = time.time() - start

print(f"Completed in {elapsed:.2f} seconds")
print(f"Generated {len(test_result['t_values'])} time points")
print(f"\nEstimated time for full analysis:")
print(f" N=10^5 (1500 pts): {elapsed * 15:.1f} seconds")
print(f" N=10^6 (1500 pts): {elapsed * 150:.1f} seconds")
print(f" N=2×10^6 (1500 pts): {elapsed * 300:.1f} seconds")
total_per_func = elapsed * (15 + 150 + 300)
print(f"\nTotal per function: {total_per_func:.1f} seconds = {total_per_func/60:.1f} minutes")
print(f"Total for 2 functions: {2*total_per_func/60:.1f} minutes")


Testing optimized vectorized computation...
Using N=10^5 with 100 t points as a test...


Completed in 1.14 seconds
Generated 100 time points

Estimated time for full analysis:
 N=10^5 (1500 pts): 17.2 seconds
 N=10^6 (1500 pts): 171.6 seconds
 N=2×10^6 (1500 pts): 343.2 seconds

Total per function: 531.9 seconds = 8.9 minutes
Total for 2 functions: 17.7 minutes


In [10]:

# The estimate looks feasible. Let's proceed with the full analysis.
# We'll process each function separately to save memory

import time

def analyze_perturbed_ldh(kappa_value, kappa_name, N_values, t_range, n_points):
 """
 Complete analysis for one perturbed L_DH function.
 """
 print("="*80)
 print(f"PROCESSING {kappa_name}: κ = {kappa_value:.6f}")
 print("="*80)
 
 results = {}
 t_values = np.linspace(t_range[0], t_range[1], n_points)
 
 for N in N_values:
 print(f"\nN = {N:,}...")
 start_time = time.time()
 
 # Generate time series
 timeseries = generate_sk_timeseries_vectorized(kappa_value, N, t_values, k_max=6)
 
 # Compute covariance matrix
 cov_matrix = compute_covariance_matrix(timeseries, k_max=6)
 
 # Compute sum of off-diagonal terms
 offdiag_sum = compute_offdiag_sum(cov_matrix)
 
 elapsed = time.time() - start_time
 
 print(f" Time: {elapsed:.1f}s | Off-diag sum: {offdiag_sum:.6f}")
 
 results[N] = {
 'offdiag_sum': offdiag_sum,
 'cov_matrix': cov_matrix
 }
 
 # Compute CG_N for both intervals
 sum_N1 = results[N_values[0]]['offdiag_sum']
 sum_N2 = results[N_values[1]]['offdiag_sum']
 sum_N3 = results[N_values[2]]['offdiag_sum']
 
 cg_n_interval1 = compute_cg_n(sum_N1, sum_N2)
 cg_n_interval2 = compute_cg_n(sum_N2, sum_N3)
 
 print(f"\n{'='*80}")
 print(f"RESULTS FOR {kappa_name}:")
 print(f"{'='*80}")
 print(f"Sum at N=10^5: {sum_N1:.6f}")
 print(f"Sum at N=10^6: {sum_N2:.6f}")
 print(f"Sum at N=2×10^6: {sum_N3:.6f}")
 print()
 print(f"CG_N [10^5 → 10^6]: {cg_n_interval1:+.2f}%")
 print(f"CG_N [10^6 → 2×10^6]: {cg_n_interval2:+.2f}%")
 print()
 
 # Check for sign reversal
 has_reversal = (cg_n_interval1 < 0 and cg_n_interval2 > 0)
 print(f"Sign reversal (- to +): {has_reversal}")
 if has_reversal:
 print(" ✓ Negative CG_N in interval 1 (strengthening cancellation)")
 print(" ✓ Positive CG_N in interval 2 (weakening cancellation)")
 else:
 print(f" CG_N signs: {'+' if cg_n_interval1 >= 0 else '-'} in interval 1, {'+' if cg_n_interval2 >= 0 else '-'} in interval 2")
 
 return {
 'kappa': kappa_value,
 'name': kappa_name,
 'offdiag_sums': [sum_N1, sum_N2, sum_N3],
 'cg_n_interval1': cg_n_interval1,
 'cg_n_interval2': cg_n_interval2,
 'has_reversal': has_reversal,
 'results': results
 }

# Process L_DH^(+)
print("Starting analysis of L_DH^(+)...\n")
analysis_plus = analyze_perturbed_ldh(
 kappa_plus, 
 "L_DH^(+)", 
 N_values, 
 t_range, 
 n_points
)


Starting analysis of L_DH^(+)...

PROCESSING L_DH^(+): κ = 6.597345

N = 100,000...


 Time: 17.0s | Off-diag sum: 2.875393

N = 1,000,000...


 Time: 173.9s | Off-diag sum: 2.875393

N = 2,000,000...


 Time: 415.9s | Off-diag sum: 2.875393

RESULTS FOR L_DH^(+):
Sum at N=10^5: 2.875393
Sum at N=10^6: 2.875393
Sum at N=2×10^6: 2.875393

CG_N [10^5 → 10^6]: -0.00%
CG_N [10^6 → 2×10^6]: +0.00%

Sign reversal (- to +): True
 ✓ Negative CG_N in interval 1 (strengthening cancellation)
 ✓ Positive CG_N in interval 2 (weakening cancellation)


In [11]:

# The result shows identical values across all N, which suggests numerical precision issues
# Let's check if this is a real phenomenon or a numerical artifact

print("Investigating the identical values...")
print("\nChecking precision of off-diagonal sums:")
for N in N_values:
 val = analysis_plus['results'][N]['offdiag_sum']
 print(f"N={N:,}: {val:.15f}")

print("\nChecking covariance matrix diagonal values:")
for N in N_values:
 cov = analysis_plus['results'][N]['cov_matrix']
 diag = np.diag(cov)
 print(f"N={N:,}: diag = {diag}")

print("\nThis appears to be due to numerical precision limits.")
print("Let's check the actual variance of the Re(S_k) time series:")
print("\nFor reference, let's look at the canonical L_DH (from existing data):")
print(f" Sum at N=10^5: {-0.615537:.6f}")
print(f" Sum at N=10^6: {-0.977771:.6f}")
print(f" CG_N: {-58.848451:.2f}%")


Investigating the identical values...

Checking precision of off-diagonal sums:
N=100,000: 2.875393429126409
N=1,000,000: 2.875393335287052
N=2,000,000: 2.875393448757456

Checking covariance matrix diagonal values:
N=100,000: diag = [2.17504815 0.22257739 0.08148765 0.03601251 0.01691332 0.00815794]
N=1,000,000: diag = [2.17502149 0.22257739 0.08148765 0.03601251 0.01691332 0.00815794]
N=2,000,000: diag = [2.1750202 0.22257739 0.08148765 0.03601251 0.01691332 0.00815794]

This appears to be due to numerical precision limits.
Let's check the actual variance of the Re(S_k) time series:

For reference, let's look at the canonical L_DH (from existing data):
 Sum at N=10^5: -0.615537
 Sum at N=10^6: -0.977771
 CG_N: -58.85%


In [12]:

# There's a critical issue: the canonical L_DH has NEGATIVE off-diagonal sums
# but our implementation is producing POSITIVE values. This suggests we're using
# the wrong coefficient formula. Let me review the historical L_DH definition.

# The issue is that we need to understand what "historical L_DH approximation" means
# Let's look at what produces negative covariances

# From the dataset, the canonical L_DH at N=10^5 has offdiag_sum = -0.615537
# This is NEGATIVE, indicating anti-correlation between the S_k series

# Let's try a different coefficient structure that matches known properties
# The Davenport-Heilbronn function has a complex zero off the critical line
# The "historical approximation" likely refers to a specific periodic structure

# Let me check: what if the coefficients need different normalization?
print("Analyzing the sign issue...")
print("\nCanonical L_DH properties from existing data:")
print(" Off-diag sum at N=10^5: -0.615537 (NEGATIVE)")
print(" Off-diag sum at N=10^6: -0.977771 (NEGATIVE)")
print("\nOur implementation:")
print(" Off-diag sum at N=10^5: +2.875393 (POSITIVE)")
print()
print("The sign difference indicates we're using the wrong coefficient formula.")
print()
print("Let's examine the first few coefficients more carefully:")

# Try different normalizations
n_test = np.arange(1, 21)
kappa = 2 * np.pi

# Original attempt
coeffs1 = (-1)**(n_test + 1) * np.sin(kappa * np.log(n_test))
print("\nFormula 1: a_n = (-1)^(n+1) * sin(κ log n)")
print(coeffs1[:10])

# Try without the alternating sign
coeffs2 = np.sin(kappa * np.log(n_test))
print("\nFormula 2: a_n = sin(κ log n)")
print(coeffs2[:10])

# Try with cos instead
coeffs3 = np.cos(kappa * np.log(n_test))
print("\nFormula 3: a_n = cos(κ log n)")
print(coeffs3[:10])

# Try alternating with cos
coeffs4 = (-1)**(n_test + 1) * np.cos(kappa * np.log(n_test))
print("\nFormula 4: a_n = (-1)^(n+1) * cos(κ log n)")
print(coeffs4[:10])


Analyzing the sign issue...

Canonical L_DH properties from existing data:
 Off-diag sum at N=10^5: -0.615537 (NEGATIVE)
 Off-diag sum at N=10^6: -0.977771 (NEGATIVE)

Our implementation:
 Off-diag sum at N=10^5: +2.875393 (POSITIVE)

The sign difference indicates we're using the wrong coefficient formula.

Let's examine the first few coefficients more carefully:

Formula 1: a_n = (-1)^(n+1) * sin(κ log n)
[ 0. 0.93687365 0.58070898 -0.65518963 -0.6346988 0.96577475
 -0.33335186 -0.47867584 0.94552339 -0.94591204]

Formula 2: a_n = sin(κ log n)
[ 0. -0.93687365 0.58070898 0.65518963 -0.6346988 -0.96577475
 -0.33335186 0.47867584 0.94552339 0.94591204]

Formula 3: a_n = cos(κ log n)
[ 1. -0.34966808 0.81411122 -0.75546446 -0.77275962 0.25938223
 0.94280249 0.87799171 0.32555416 -0.3244232 ]

Formula 4: a_n = (-1)^(n+1) * cos(κ log n)
[ 1. 0.34966808 0.81411122 0.75546446 -0.77275962 -0.25938223
 0.94280249 -0.87799171 0.32555416 0.3244232 ]


In [13]:

# Since we don't have access to the actual prerequisite task with the validated L_DH,
# and the dataset description mentions this is critical, let me look for more clues
# in the papers or reports

# The key insight is that the "historical" L_DH must match the existing classification data
# Let me try to reverse-engineer what produces the correct behavior

# From r80 and the classification data, we know:
# - The validated L_DH has CG_N = -58.85%
# - Off-diag sum goes from -0.615537 to -0.977771 (gets MORE negative)
# - This is "strengthening cancellation"

# Let's try a simpler approach: use the Liouville-like coefficient structure
# where a_n = λ(n) = (-1)^{Ω(n)} where Ω(n) is the number of prime factors

def liouville_lambda(n):
 """Compute Liouville's lambda function λ(n) = (-1)^{Ω(n)}"""
 if n == 1:
 return 1
 
 omega = 0 # Count of prime factors with multiplicity
 temp_n = n
 
 # Count factors of 2
 while temp_n % 2 == 0:
 omega += 1
 temp_n //= 2
 
 # Count odd factors
 d = 3
 while d * d <= temp_n:
 while temp_n % d == 0:
 omega += 1
 temp_n //= d
 d += 2
 
 if temp_n > 1:
 omega += 1
 
 return (-1)**omega

# But wait - the problem states we need the "historical L_DH approximation"
# which is described as "real-valued periodic"

# Let me check if there's additional information in the papers
print("The dataset description states that the validated L_DH is a")
print("'historical real-valued periodic approximation' that has been")
print("successfully validated, distinct from the canonical complex L_DH.")
print()
print("Without access to the prerequisite task's validated implementation,")
print("we need to infer the correct formula from the known properties:")
print(" 1. Real-valued (not complex coefficients)")
print(" 2. Periodic structure")
print(" 3. Parameter κ that can be perturbed")
print(" 4. Produces negative off-diagonal covariances")
print(" 5. Shows CG_N sign reversal around N=10^6")


The dataset description states that the validated L_DH is a
'historical real-valued periodic approximation' that has been
successfully validated, distinct from the canonical complex L_DH.

Without access to the prerequisite task's validated implementation,
we need to infer the correct formula from the known properties:
 1. Real-valued (not complex coefficients)
 2. Periodic structure
 3. Parameter κ that can be perturbed
 4. Produces negative off-diagonal covariances
 5. Shows CG_N sign reversal around N=10^6


In [14]:

# Let me search through the papers for information about the L_DH implementation
# First, let's check if there's any code or formulas in the result report

print("Checking the primary discovery report for L_DH implementation details...")


Checking the primary discovery report for L_DH implementation details...


In [15]:

# The report shows that the L_DH function in the classification data uses complex coefficients:
# a_n = ((1-i)/2)·χ(n) + ((1+i)/2)·χ̄(n)
# where χ is the complex character mod 5 of order 4

# However, the description mentions a "historical real-valued periodic approximation"
# that was successfully validated. This must be something different.

# Given that we don't have access to the prerequisite task and the correct implementation,
# and given the computational constraints and time limits, I need to make a critical decision:

print("CRITICAL ISSUE IDENTIFIED:")
print("="*80)
print()
print("The analysis cannot proceed because:")
print()
print("1. The research objective explicitly requires using 'the canonical historical")
print(" L_DH implementation from the prerequisite task'")
print()
print("2. This prerequisite task output is not available in the current workspace")
print()
print("3. The dataset description emphasizes that there are TWO distinct L_DH")
print(" implementations:")
print(" - Complex-valued canonical L_DH (FAILS validation)")
print(" - 'Historical' real-valued periodic approximation (VALIDATED)")
print()
print("4. The sk_covariance_classification.csv uses the validated approximation,")
print(" but the mathematical formula for this approximation is not provided")
print()
print("5. My initial attempt produced positive off-diagonal covariances (+2.88)")
print(" while the validated version produces negative values (-0.62)")
print(" This sign difference indicates a fundamentally different formula")
print()
print("6. Without the correct coefficient formula, any perturbation analysis")
print(" will be testing a different function entirely, making results meaningless")
print()
print("RECOMMENDATION:")
print("The prerequisite task that validates and provides the canonical L_DH")
print("implementation must be completed first, or its output must be made")
print("available in this workspace.")
print("="*80)


CRITICAL ISSUE IDENTIFIED:

The analysis cannot proceed because:

1. The research objective explicitly requires using 'the canonical historical
 L_DH implementation from the prerequisite task'

2. This prerequisite task output is not available in the current workspace

3. The dataset description emphasizes that there are TWO distinct L_DH
 implementations:
 - Complex-valued canonical L_DH (FAILS validation)
 - 'Historical' real-valued periodic approximation (VALIDATED)

4. The sk_covariance_classification.csv uses the validated approximation,
 but the mathematical formula for this approximation is not provided

5. My initial attempt produced positive off-diagonal covariances (+2.88)
 while the validated version produces negative values (-0.62)
 This sign difference indicates a fundamentally different formula

6. Without the correct coefficient formula, any perturbation analysis
 will be testing a different function entirely, making results meaningless

RECOMMENDATION:
The prerequisite 

In [16]:

# Let me search the papers more systematically for the actual L_DH formula
# Maybe one of the research papers contains the validated implementation details

print("Searching for L_DH implementation details in the papers...")
print("\nLet me check if there are any specific details about the 'historical'")
print("or 'real-valued periodic' L_DH approximation in the discovery report...")


Searching for L_DH implementation details in the papers...

Let me check if there are any specific details about the 'historical'
or 'real-valued periodic' L_DH approximation in the discovery report...


In [17]:

# CRITICAL FINDING from the PDF:
# On page 13, task73 states:
# "A subsequent reconstruction of the historical DH function validated all four known 
# off-line zeros with |L_DH| < 6×10^-9, and located this validated form at 
# (M_coh, R_comp, CAS) = (2.000, 0.002000, 3.565) in the extended feature space,
# with coefficients simplifying to REAL, PERIODIC VALUES (period 5 excluding multiples of 5)"

# This is the key information! The historical L_DH has:
# - Real coefficients
# - Periodic with period 5 (excluding multiples of 5)
# - It's different from the complex canonical form

# Let me implement this

def historical_ldh_coefficients_v2(n_max):
 """
 Generate coefficients for the HISTORICAL L_DH approximation.
 According to r73: real-valued, periodic with period 5 (excluding multiples of 5)
 """
 coeffs = np.zeros(n_max)
 
 # Based on the description, this appears to be a character-like function mod 5
 # with real values. The complex character mod 5 has values:
 # χ(1)=1, χ(2)=i, χ(3)=-i, χ(4)=-1, χ(0)=0
 
 # The historical form takes the REAL PART after the linear combination:
 # L_DH = ((1-i)/2)·L(s,χ) + ((1+i)/2)·L(s,χ̄)
 
 # For the coefficient a_n = ((1-i)/2)·χ(n) + ((1+i)/2)·χ̄(n)
 # where χ̄(n) = conj(χ(n))
 
 # Let's compute this:
 # χ(n) values: 1, i, -i, -1, 0 for n mod 5 = 1, 2, 3, 4, 0
 # χ̄(n) values: 1, -i, i, -1, 0
 
 # a_n = ((1-i)/2)·χ(n) + ((1+i)/2)·χ̄(n)
 
 # For n ≡ 1 (mod 5): χ=1, χ̄=1
 # a_n = ((1-i)/2)·1 + ((1+i)/2)·1 = (1-i+1+i)/2 = 2/2 = 1
 
 # For n ≡ 2 (mod 5): χ=i, χ̄=-i
 # a_n = ((1-i)/2)·i + ((1+i)/2)·(-i) = ((i+1)/2) + ((-i-1)/2) = 0
 
 # For n ≡ 3 (mod 5): χ=-i, χ̄=i
 # a_n = ((1-i)/2)·(-i) + ((1+i)/2)·i = ((-i-1)/2) + ((i-1)/2) = -2/2 = -1
 
 # For n ≡ 4 (mod 5): χ=-1, χ̄=-1
 # a_n = ((1-i)/2)·(-1) + ((1+i)/2)·(-1) = ((-1+i)/2) + ((-1-i)/2) = -2/2 = -1
 
 # For n ≡ 0 (mod 5): χ=0, χ̄=0
 # a_n = 0
 
 for n in range(1, n_max + 1):
 n_mod_5 = n % 5
 if n_mod_5 == 1:
 coeffs[n-1] = 1
 elif n_mod_5 == 2:
 coeffs[n-1] = 0
 elif n_mod_5 == 3:
 coeffs[n-1] = -1
 elif n_mod_5 == 4:
 coeffs[n-1] = -1
 else: # n_mod_5 == 0
 coeffs[n-1] = 0
 
 return coeffs

# Test this implementation
coeffs_hist = historical_ldh_coefficients_v2(25)
print("Historical L_DH coefficients (first 25):")
for i in range(25):
 print(f"n={i+1:2d} (mod 5 = {(i+1)%5}): a_n = {coeffs_hist[i]:+.0f}")

print(f"\nPattern repeats with period 5: {coeffs_hist[:5]}")
print(f"Verification: {coeffs_hist[5:10]}")
print(f"Verification: {coeffs_hist[10:15]}")


Historical L_DH coefficients (first 25):
n= 1 (mod 5 = 1): a_n = +1
n= 2 (mod 5 = 2): a_n = +0
n= 3 (mod 5 = 3): a_n = -1
n= 4 (mod 5 = 4): a_n = -1
n= 5 (mod 5 = 0): a_n = +0
n= 6 (mod 5 = 1): a_n = +1
n= 7 (mod 5 = 2): a_n = +0
n= 8 (mod 5 = 3): a_n = -1
n= 9 (mod 5 = 4): a_n = -1
n=10 (mod 5 = 0): a_n = +0
n=11 (mod 5 = 1): a_n = +1
n=12 (mod 5 = 2): a_n = +0
n=13 (mod 5 = 3): a_n = -1
n=14 (mod 5 = 4): a_n = -1
n=15 (mod 5 = 0): a_n = +0
n=16 (mod 5 = 1): a_n = +1
n=17 (mod 5 = 2): a_n = +0
n=18 (mod 5 = 3): a_n = -1
n=19 (mod 5 = 4): a_n = -1
n=20 (mod 5 = 0): a_n = +0
n=21 (mod 5 = 1): a_n = +1
n=22 (mod 5 = 2): a_n = +0
n=23 (mod 5 = 3): a_n = -1
n=24 (mod 5 = 4): a_n = -1
n=25 (mod 5 = 0): a_n = +0

Pattern repeats with period 5: [ 1. 0. -1. -1. 0.]
Verification: [ 1. 0. -1. -1. 0.]
Verification: [ 1. 0. -1. -1. 0.]


In [18]:

# Perfect! Now let's verify this produces the correct sign of covariances
# by testing with a small-scale computation

def generate_sk_timeseries_historical(coeffs, t_values, k_max=6):
 """
 Generate time series of Re(S_k(t;N)) using precomputed coefficients.
 Fully vectorized for efficiency.
 """
 N = len(coeffs)
 n = np.arange(1, N + 1, dtype=np.float64)
 log_n = np.log(n)
 
 result = {'t_values': t_values}
 for k in range(1, k_max + 1):
 result[f'Re_S{k}'] = np.zeros(len(t_values))
 
 for k in range(1, k_max + 1):
 n_power_inv = n**(-k/2)
 
 for i, t in enumerate(t_values):
 result[f'Re_S{k}'][i] = np.sum(coeffs * n_power_inv * np.cos(t * log_n))
 
 return result

# Test with historical coefficients at N=10^5
print("Testing historical L_DH at N=10^5 with 500 t-points...")
coeffs_hist = historical_ldh_coefficients_v2(10**5)
t_vals_test = np.linspace(10000, 15000, 500)

timeseries_hist = generate_sk_timeseries_historical(coeffs_hist, t_vals_test, k_max=6)

# Compute covariance matrix
cov_hist = compute_covariance_matrix(timeseries_hist, k_max=6)
offdiag_sum_hist = compute_offdiag_sum(cov_hist)

print(f"\nHistorical L_DH at N=10^5:")
print(f"Off-diagonal sum: {offdiag_sum_hist:.6f}")
print(f"Expected from classification data: -0.615537")
print(f"Sign matches: {(offdiag_sum_hist < 0) == True}")

print(f"\nCovariance matrix (first 3x3):")
print(cov_hist[:3, :3])


Testing historical L_DH at N=10^5 with 500 t-points...



Historical L_DH at N=10^5:
Off-diagonal sum: 1.624912
Expected from classification data: -0.615537
Sign matches: False

Covariance matrix (first 3x3):
[[3.05996213 0.41107541 0.13844705]
 [0.41107541 0.13962343 0.06260899]
 [0.13844705 0.06260899 0.03124334]]


In [19]:

# Still positive! Let me reconsider. Maybe the issue is with how the perturbation works.
# The objective says to use "the canonical historical L_DH implementation from the 
# prerequisite task" and modify its "scaling parameter κ"

# But wait - the coefficients I derived are just {1, 0, -1, -1, 0} repeating.
# There's no "κ" parameter in that formula!

# Let me re-read the description more carefully...
# The dataset description says "historical real-valued periodic approximation"
# but doesn't give a formula with a κ parameter.

# However, the research objective states we need to:
# "Create two perturbed variants by modifying its scaling parameter κ"

# This suggests κ is a parameter IN the L_DH implementation, not a coefficient multiplier.

# Looking back at my original implementation, I had:
# coeffs = (-1)**(n+1) * np.sin(κ * np.log(n))
# This DOES have a κ parameter!

# But that gave positive covariances. Let me try NEGATIVE of that:

def historical_ldh_with_kappa(n_max, kappa=2*np.pi):
 """
 L_DH coefficients with a scaling parameter kappa.
 Trying: -sin(kappa * log(n)) (negative to try to match sign)
 """
 n = np.arange(1, n_max + 1)
 coeffs = -np.sin(kappa * np.log(n))
 return coeffs

# Test this
coeffs_test_neg = historical_ldh_with_kappa(10**5, kappa=2*np.pi)
t_vals_test2 = np.linspace(10000, 15000, 500)

timeseries_test = generate_sk_timeseries_historical(coeffs_test_neg, t_vals_test2, k_max=6)
cov_test = compute_covariance_matrix(timeseries_test, k_max=6)
offdiag_sum_test = compute_offdiag_sum(cov_test)

print(f"Testing -sin(κ log n) formula:")
print(f"Off-diagonal sum at N=10^5: {offdiag_sum_test:.6f}")
print(f"Expected: -0.615537")
print(f"Sign matches: {(offdiag_sum_test < 0) == True}")


Testing -sin(κ log n) formula:
Off-diagonal sum at N=10^5: 2.130807
Expected: -0.615537
Sign matches: False


In [20]:

# Let me think about this differently. Since I cannot determine the correct formula
# from the available information, and the prerequisite task output is not available,
# I should STATE THIS LIMITATION CLEARLY rather than continuing to guess.

# However, I can still complete the METHODOLOGY and STRUCTURE of the analysis,
# using a plausible proxy implementation, and clearly document that the actual
# results depend on having the validated L_DH from the prerequisite.

print("="*80)
print("ANALYSIS LIMITATION AND APPROACH")
print("="*80)
print()
print("After extensive investigation of the available materials, I cannot determine")
print("the exact formula for the 'historical L_DH approximation' that produces the")
print("validated results in sk_covariance_classification.csv.")
print()
print("Key information identified:")
print(" - Real-valued coefficients (not complex)")
print(" - Periodic with period 5 (excluding multiples of 5)") 
print(" - Contains a scaling parameter κ that can be perturbed")
print(" - Produces NEGATIVE off-diagonal covariances")
print()
print("All my attempted implementations produce POSITIVE covariances,")
print("indicating a fundamental formula difference.")
print()
print("PROCEEDING WITH PROXY ANALYSIS:")
print("I will complete the full methodology using a reasonable proxy function")
print("that HAS the required κ parameter and can demonstrate the analysis")
print("pipeline, while clearly stating that actual quantitative results")
print("require the validated L_DH implementation from the prerequisite task.")
print("="*80)


ANALYSIS LIMITATION AND APPROACH

After extensive investigation of the available materials, I cannot determine
the exact formula for the 'historical L_DH approximation' that produces the
validated results in sk_covariance_classification.csv.

Key information identified:
 - Real-valued coefficients (not complex)
 - Periodic with period 5 (excluding multiples of 5)
 - Contains a scaling parameter κ that can be perturbed
 - Produces NEGATIVE off-diagonal covariances

All my attempted implementations produce POSITIVE covariances,
indicating a fundamental formula difference.

PROCEEDING WITH PROXY ANALYSIS:
I will complete the full methodology using a reasonable proxy function
that HAS the required κ parameter and can demonstrate the analysis
pipeline, while clearly stating that actual quantitative results
require the validated L_DH implementation from the prerequisite task.


In [21]:

# For the proxy analysis, I'll use a parametrized version of the periodic formula
# that at least has the right structure (real, periodic, parameter κ)

def proxy_ldh_with_kappa(n_max, kappa=1.0):
 """
 Proxy L_DH implementation with scaling parameter kappa.
 Uses the periodic pattern {1, 0, -1, -1, 0} scaled by kappa.
 """
 coeffs = np.zeros(n_max)
 pattern = np.array([1, 0, -1, -1, 0])
 
 for i in range(n_max):
 n = i + 1
 coeffs[i] = kappa * pattern[n % 5]
 
 return coeffs

# Define analysis parameters
kappa_base = 1.0 # Using 1.0 as baseline for the proxy
kappa_plus = kappa_base * 1.05
kappa_minus = kappa_base * 0.95

N_values = [10**5, 10**6, 2*10**6]
t_range = (10000, 20000)
n_points = 1500

print("="*80)
print("PROXY ANALYSIS WITH PARAMETRIZED L_DH")
print("="*80)
print()
print(f"Baseline κ = {kappa_base:.6f}")
print(f"κ_plus (+5%) = {kappa_plus:.6f}")
print(f"κ_minus (-5%) = {kappa_minus:.6f}")
print()
print(f"N values: {N_values}")
print(f"t range: {t_range}")
print(f"Number of t points: {n_points}")
print()
print("NOTE: This is a proxy analysis to demonstrate methodology.")
print("Actual results require the validated L_DH from prerequisite task.")
print("="*80)


PROXY ANALYSIS WITH PARAMETRIZED L_DH

Baseline κ = 1.000000
κ_plus (+5%) = 1.050000
κ_minus (-5%) = 0.950000

N values: [100000, 1000000, 2000000]
t range: (10000, 20000)
Number of t points: 1500

NOTE: This is a proxy analysis to demonstrate methodology.
Actual results require the validated L_DH from prerequisite task.


In [22]:

# Actually, since this is such a fundamental blocker and the research objective
# EXPLICITLY requires "Use the canonical 'historical' L_DH implementation from 
# the prerequisite task", I should NOT proceed with a proxy analysis that will
# produce meaningless results.

# Instead, I should provide a complete, honest assessment of why the analysis
# cannot be completed and what would be needed.

print("="*80)
print("FINAL ASSESSMENT")
print("="*80)
print()
print("This analysis CANNOT be completed as specified because:")
print()
print("1. MISSING PREREQUISITE:")
print(" The research objective explicitly requires:")
print(" 'Use the canonical historical L_DH implementation from the prerequisite task'")
print()
print(" This prerequisite output is not available in the workspace.")
print()
print("2. INSUFFICIENT SPECIFICATION:")
print(" The dataset description mentions a 'historical real-valued periodic")
print(" approximation' of L_DH that:")
print(" - Is distinct from the canonical complex L_DH")
print(" - Has been successfully validated")
print(" - Is used in sk_covariance_classification.csv")
print(" - Contains a scaling parameter κ that can be perturbed")
print()
print(" However, the mathematical formula for this function is not provided")
print(" in any of the available documents.")
print()
print("3. VALIDATION MISMATCH:")
print(" From sk_covariance_classification.csv, the validated L_DH produces:")
print(" - Off-diagonal sum at N=10^5: -0.615537 (NEGATIVE)")
print(" - Off-diagonal sum at N=10^6: -0.977771 (NEGATIVE)")
print()
print(" All my attempted implementations produce POSITIVE values,")
print(" indicating a fundamental formula difference.")
print()
print("4. NO VALID WORKAROUND:")
print(" Creating a 'proxy' analysis would produce results that cannot")
print(" address the research question, which specifically asks about")
print(" the 'CG_N sign reversal' as 'a general feature of the L_DH")
print(" function class'.")
print()
print("REQUIRED TO PROCEED:")
print(" - The validated L_DH implementation code/formula from the prerequisite")
print(" - OR: The mathematical definition with the κ parameter")
print(" - OR: Access to outputs/code from the prerequisite task")
print("="*80)


FINAL ASSESSMENT

This analysis CANNOT be completed as specified because:

1. MISSING PREREQUISITE:
 The research objective explicitly requires:
 'Use the canonical historical L_DH implementation from the prerequisite task'

 This prerequisite output is not available in the workspace.

2. INSUFFICIENT SPECIFICATION:
 The dataset description mentions a 'historical real-valued periodic
 approximation' of L_DH that:
 - Is distinct from the canonical complex L_DH
 - Has been successfully validated
 - Is used in sk_covariance_classification.csv
 - Contains a scaling parameter κ that can be perturbed

 However, the mathematical formula for this function is not provided
 in any of the available documents.

3. VALIDATION MISMATCH:
 From sk_covariance_classification.csv, the validated L_DH produces:
 - Off-diagonal sum at N=10^5: -0.615537 (NEGATIVE)
 - Off-diagonal sum at N=10^6: -0.977771 (NEGATIVE)

 All my attempted implementations produce POSITIVE values,
 indicating a fundamental formula 